In [4]:
# ================================
# 1. Import Libraries
# ================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, RocCurveDisplay)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import joblib


# ================================
# 2. Load Processed Dataset
# ================================
X = pd.read_csv("../model/X_scaled.csv")
y = pd.read_csv("../model/y.csv")
y = y.values.ravel()

print("Feature shape:", X.shape)
print("Target shape :", y.shape)
print("Churn rate   :", round(y.mean() * 100, 2), "%")


# ================================
# 3. Train Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Test Data    :", X_test.shape)


# ================================
# 4. Handle Class Imbalance (SMOTE)
# ================================
print("\nClass distribution BEFORE SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

print("Class distribution AFTER SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))


# ================================
# 5. Cross Validation Setup
# ================================
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ================================
# 6. Logistic Regression Model
# ================================
print("\n--- Training Logistic Regression ---")

log_model = LogisticRegression(max_iter=1000, class_weight='balanced')

log_params = {
    "C": [0.01, 0.1, 1, 10]
}

log_grid = GridSearchCV(
    log_model,
    log_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

log_grid.fit(X_train, y_train)
best_log_model = log_grid.best_estimator_
print("Best Logistic Parameters:", log_grid.best_params_)


# ================================
# 7. Random Forest Model
# ================================
print("\n--- Training Random Forest ---")

rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'       # <-- ADDED
)

rf_params = {
    "n_estimators"    : [100, 200],
    "max_depth"       : [5, 10, None],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf_model,
    rf_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)
best_rf_model = rf_grid.best_estimator_
print("Best RF Parameters:", rf_grid.best_params_)


# ================================
# 8. XGBoost Model
# ================================
print("\n--- Training XGBoost ---")

xgb_model = XGBClassifier(
    eval_metric="logloss",
    use_label_encoder=False,
    scale_pos_weight=3            # <-- ADDED (handles imbalance for XGB)
)

xgb_params = {
    "n_estimators" : [100, 200],
    "max_depth"    : [3, 5],
    "learning_rate": [0.01, 0.1]
}

xgb_grid = GridSearchCV(
    xgb_model,
    xgb_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)
best_xgb_model = xgb_grid.best_estimator_
print("Best XGB Parameters:", xgb_grid.best_params_)


# ================================
# 9. Evaluate Model Function
# ================================
def evaluate_model(model, X_test, y_test, model_name="Model"):

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}   <-- key metric for churn")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC AUC   : {auc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred))

    print("  Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(f"  TN={cm[0][0]}  FP={cm[0][1]}")
    print(f"  FN={cm[1][0]}  TP={cm[1][1]}")

    # Save ROC curve plot
    fig, ax = plt.subplots(figsize=(6, 4))
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=model_name)
    ax.set_title(f"ROC Curve - {model_name}")
    plt.tight_layout()
    filename = model_name.lower().replace(" ", "_")
    plt.savefig(f"../model/roc_{filename}.png", dpi=150)
    plt.close()
    print(f"  ROC curve saved → roc_{filename}.png")

    return {
        "model"    : model_name,
        "accuracy" : acc,
        "precision": prec,
        "recall"   : rec,
        "f1"       : f1,
        "roc_auc"  : auc
    }


# ================================
# 10. Evaluate All Models
# ================================
results = []
results.append(evaluate_model(best_log_model, X_test, y_test, "Logistic Regression"))
results.append(evaluate_model(best_rf_model,  X_test, y_test, "Random Forest"))
results.append(evaluate_model(best_xgb_model, X_test, y_test, "XGBoost"))


# ================================
# 11. Model Comparison Table
# ================================
print("\n")
print("=" * 70)
print(f"{'MODEL COMPARISON SUMMARY':^70}")
print("=" * 70)
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>7} {'ROC-AUC':>9}")
print("-" * 70)
for r in results:
    print(f"{r['model']:<22} {r['accuracy']:>9.4f} {r['precision']:>10.4f} "
          f"{r['recall']:>8.4f} {r['f1']:>7.4f} {r['roc_auc']:>9.4f}")
print("=" * 70)


# ================================
# 12. Feature Importance Plot
# ================================
print("\n--- Feature Importance (Random Forest) ---")

feature_importance = pd.Series(
    best_rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

# Save plot
fig, ax = plt.subplots(figsize=(8, 5))
feature_importance.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("Top 10 Important Features - Random Forest")
ax.set_xlabel("Importance Score")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../model/feature_importance.png", dpi=150)
plt.close()
print("Saved → feature_importance.png")


# ================================
# 13. Save Best Model (auto-pick)
# ================================
best_result = max(results, key=lambda x: x['recall'])

model_map = {
    "Logistic Regression": best_log_model,
    "Random Forest"      : best_rf_model,
    "XGBoost"            : best_xgb_model
}

final_model = model_map[best_result['model']]

print(f"\nBest Model  : {best_result['model']}")
print(f"ROC-AUC     : {best_result['roc_auc']:.4f}")
print(f"Recall      : {best_result['recall']:.4f}")

joblib.dump(final_model, "../model/churn_model.pkl")
print("Saved → churn_model.pkl")


# ================================
# 14. Verify Saved Model
# ================================
model = joblib.load("../model/churn_model.pkl")
print("\nModel loaded successfully!")
print("Model type:", type(model).__name__)

Feature shape: (7043, 30)
Target shape : (7043,)
Churn rate   : 26.54 %

Training Data: (5634, 30)
Test Data    : (1409, 30)

Class distribution BEFORE SMOTE:
{np.int64(0): np.int64(4139), np.int64(1): np.int64(1495)}
Class distribution AFTER SMOTE:
{np.int64(0): np.int64(4139), np.int64(1): np.int64(4139)}

--- Training Logistic Regression ---
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Logistic Parameters: {'C': 10}

--- Training Random Forest ---
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best RF Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

--- Training XGBoost ---
Fitting 5 folds for each of 8 candidates, totalling 40 fits


c:\Users\USER\Desktop\customer churn prediction\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:39:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best XGB Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}

  Logistic Regression
  Accuracy  : 0.7495
  Precision : 0.5206
  Recall    : 0.7086   <-- key metric for churn
  F1 Score  : 0.6002
  ROC AUC   : 0.8249

  Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.76      0.82      1035
           1       0.52      0.71      0.60       374

    accuracy                           0.75      1409
   macro avg       0.70      0.74      0.71      1409
weighted avg       0.78      0.75      0.76      1409

  Confusion Matrix:
  TN=791  FP=244
  FN=109  TP=265
  ROC curve saved → roc_logistic_regression.png

  Random Forest
  Accuracy  : 0.7793
  Precision : 0.5748
  Recall    : 0.6471   <-- key metric for churn
  F1 Score  : 0.6088
  ROC AUC   : 0.8343

  Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.83      0.85      1035
           1       0.57  

In [5]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install xgboost imbalanced-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
